<a href="https://colab.research.google.com/github/egxl/Turnitin_Similaritas_P3MD/blob/main/Turnitin_Similaritas_P3MD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔍 Pemeriksa Similaritas Turnitin-Style Dokumen P3MD
Notebook ini mereplikasi mekanisme resmi **Turnitin** dalam mendeteksi similaritas teks antardokumen:
- **Metode String Matching & K-Gram Shingling:** Mencocokkan deretan kata berurutan (*verbatim string matching*).
- **Kalkulasi Word-Level Similarity Index:** Menghitung rasio jumlah kata pada bagian yang cocok terhadap total kata dokumen (sesuai standar resmi Turnitin).
- **Penggabungan Rentang Teks (*Passage Reconstruction*):** Menggabungkan kata-kata yang cocok menjadi potongan kalimat utuh seperti *highlight* Turnitin.
- **Fitur Eksklusi Turnitin:** Opsi mengabaikan Daftar Pustaka (*Bibliography*) dan teks kutipan langsung dalam tanda petik (*Quotes*).
- **Standar Warna Turnitin:** 🔵 Blue (0%), 🟢 Green (1–24%), 🟡 Yellow (25–49%), 🟠 Orange (50–74%), 🔴 Red (75–100%).

### 📌 Langkah Cepat:
1. Tempelkan link folder Google Drive di sel **Langkah 2**.
2. Klik menu **Runtime > Run all** ().
3. Laporan Excel interaktif akan otomatis diunduh.

In [ ]:
# @title 1. Instalasi Library Pendukung
!pip install -q python-docx pypdf gdown openpyxl pandas
print("✅ Library siap digunakan.")

In [ ]:
# @title 2. Konfigurasi Analisis (Sesuai Parameter Turnitin) { run: "auto" }
import os, re, shutil

# Tempelkan link folder Google Drive tugas di sini:
drive_folder_url = "https://drive.google.com/drive/u/0/folders/1YbKgSou6XhmCr1CLD2dRWy_ahzQ3HFDi"  # @param {type:"string"}

# Ambang batas minimal kata berurutan yang dianggap plagiasi (Standar Turnitin: 6 - 8 kata)
min_consecutive_words = 6  # @param {type:"slider", min:4, max:12, step:1}

# Eksklusi Turnitin:
exclude_quotes = True  # @param {type:"boolean"}
exclude_bibliography = True  # @param {type:"boolean"}

# Batas preview kalimat cocok yang dicetak di layar
max_preview_passages = 5  # @param {type:"integer"}

print(f"""⚙️ Parameter Aktif:
- Min Words Match: {min_consecutive_words} kata berurutan
- Exclude Quotes: {exclude_quotes}
- Exclude Bibliography: {exclude_bibliography}""")

In [ ]:
# @title 3. Download Dokumen (Drive API - Unlimited Files)
import io
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

# Authenticate user
auth.authenticate_user()
drive_service = build('drive', 'v3')

output_dir = "./dokumen_tugas_p3md"
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)
os.makedirs(output_dir, exist_ok=True)

def extract_folder_id(url):
    match = re.search(r"folders/([a-zA-Z0-9_-]+)", url)
    if match: return match.group(1)
    match = re.search(r"id=([a-zA-Z0-9_-]+)", url)
    if match: return match.group(1)
    return url.strip()

def download_large_folder(folder_id, destination):
    query = f"'{folder_id}' in parents and trashed = false"
    results = drive_service.files().list(q=query, fields="files(id, name, mimeType)").execute()
    items = results.get('files', [])

    if not items:
        print("Empty folder found.")
        return

    print(f"📥 Found {len(items)} items. Starting download...")

    for item in items:
        file_id = item['id']
        file_name = item['name']
        mime_type = item['mimeType']

        # Skip sub-folders for simplicity in this flat analysis
        if mime_type == 'application/vnd.google-apps.folder':
            continue

        request = drive_service.files().get_media(fileId=file_id)
        fh = io.FileIO(os.path.join(destination, file_name), 'wb')
        downloader = MediaIoBaseDownload(fh, request)
        done = False
        while done is False:
            status, done = downloader.next_chunk()
        print(f"✅ Downloaded: {file_name}")

if not drive_folder_url.strip():
    print("⚠️ drive_folder_url masih kosong!")
else:
    f_id = extract_folder_id(drive_folder_url)
    try:
        download_large_folder(f_id, output_dir)
        downloaded_count = len([f for f in os.listdir(output_dir) if not f.startswith(".")])
        print(f"\n🎉 Total {downloaded_count} dokumen berhasil diunduh.")
    except Exception as e:
        print(f"❌ Gagal mengunduh via API: {e}")

In [ ]:
# @title 4. Turnitin Matching Engine (Word-Level Containment)
from itertools import combinations
import pandas as pd
from docx import Document
from pypdf import PdfReader
import os
import re

def extract_raw_text(file_path):
    ext = os.path.splitext(file_path)[1].lower()
    text = ""
    try:
        if ext == ".docx":
            doc = Document(file_path)
            text = "\n".join([p.text for p in doc.paragraphs if p.text.strip()])
        elif ext == ".pdf":
            reader = PdfReader(file_path)
            text = "\n".join([page.extract_text() or "" for page in reader.pages])
        elif ext == ".txt":
            with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
                text = f.read()
    except Exception as e:
        print(f"⚠️ Gagal mengekstrak {os.path.basename(file_path)}: {e}")
    return text

def apply_turnitin_exclusions(text, drop_quotes=True, drop_bib=True):
    """Menerapkan filter eksklusi Turnitin: kutipan dan daftar pustaka."""
    if drop_bib:
        bib_pattern = r"\n\s*(?:daftar\s+pustaka|references|bibliography|rujukan)\s*[:\n]"
        parts = re.split(bib_pattern, text, flags=re.IGNORECASE)
        if len(parts) > 1:
            text = parts[0]

    if drop_quotes:
        text = re.sub(r'"[^"]*"|[“”][^“”]*[“”]', ' ', text)

    return text

def tokenize_words(text):
    return re.findall(r"\b\w+\b", text.lower())

def build_kgram_map(words, k=6):
    kgram_map = {}
    for i in range(len(words) - k + 1):
        gram = " ".join(words[i:i+k])
        if gram not in kgram_map:
            kgram_map[gram] = []
        kgram_map[gram].append(i)
    return kgram_map

def calculate_turnitin_similarity(doc_a_words, doc_b_words, k=6):
    if len(doc_a_words) < k or len(doc_b_words) < k:
        return 0.0, 0.0, []

    map_a = build_kgram_map(doc_a_words, k)
    map_b = build_kgram_map(doc_b_words, k)

    common_grams = set(map_a.keys()).intersection(set(map_b.keys()))

    matched_indices_a = set()
    for gram in common_grams:
        for start_idx in map_a[gram]:
            for offset in range(k):
                matched_indices_a.add(start_idx + offset)

    matched_indices_b = set()
    for gram in common_grams:
        for start_idx in map_b[gram]:
            for offset in range(k):
                matched_indices_b.add(start_idx + offset)

    score_a = (len(matched_indices_a) / len(doc_a_words) * 100) if doc_a_words else 0.0
    score_b = (len(matched_indices_b) / len(doc_b_words) * 100) if doc_b_words else 0.0

    passages = []
    if matched_indices_a:
        sorted_indices = sorted(matched_indices_a)
        current_passage = [doc_a_words[sorted_indices[0]]]
        for prev_idx, curr_idx in zip(sorted_indices[:-1], sorted_indices[1:]):
            if curr_idx == prev_idx + 1:
                current_passage.append(doc_a_words[curr_idx])
            else:
                passages.append(" ".join(current_passage))
                current_passage = [doc_a_words[curr_idx]]
        passages.append(" ".join(current_passage))

    return score_a, score_b, passages

supported_exts = {".docx", ".pdf", ".txt"}
file_paths = []
for root, _, files in os.walk(output_dir):
    for f in files:
        if os.path.splitext(f)[1].lower() in supported_exts and not f.startswith("~"):
            file_paths.append(os.path.join(root, f))

doc_database = {}
print("📄 Membaca dan memproses dokumen dengan filter Turnitin...")
for fp in file_paths:
    name = os.path.basename(fp)
    raw = extract_raw_text(fp)
    filtered = apply_turnitin_exclusions(raw, drop_quotes=exclude_quotes, drop_bib=exclude_bibliography)
    words = tokenize_words(filtered)
    if len(words) > 0:
        doc_database[name] = words

print(f"✅ {len(doc_database)} dokumen berhasil diekstrak dan siap dihitung.")

In [ ]:
import math

# Estimasi beban kerja untuk 398 dokumen
total_docs = 398
total_pairs = math.comb(total_docs, 2)

print(f"📊 Analisis Skalabilitas untuk {total_docs} dokumen:")
print(f"- Total pasangan yang akan dibandingkan: {total_pairs:,} pasang")

avg_words = sum(len(words) for words in doc_database.values()) / len(doc_database) if doc_database else 0
print(f"- Rata-rata kata per dokumen saat ini: {int(avg_words)} kata")
print(f"- Estimasi total kata di memori: {int(avg_words * total_docs):,} kata")

if total_pairs > 50000:
    print("\n⚠️ Peringatan: Jumlah pasangan sangat besar. Proses mungkin memakan waktu beberapa menit.")
    print("💡 Tips: Jika Colab Crash, kita perlu mengubah metode penyimpanan ke generator atau database sementara.")

In [ ]:
# @title 5. Rekapitulasi Similarity Report (Optimized for 398+ Docs)
from google.colab import files
import openpyxl
from openpyxl.styles import PatternFill, Font

# Threshold dari Komandan Batch (17%), Cushioning ke 15%
COMMANDER_THRESHOLD = 17.0
PASS_THRESHOLD = 15.0

def turnitin_badge(score):
    if score == 0: return "🔵 Blue (0%)"
    elif score < 25: return "🟢 Green (1-24%)"
    elif score < 50: return "🟡 Yellow (25-49%)"
    elif score < 75: return "🟠 Orange (50-74%)"
    else: return "🔴 Red (75-100%)"

def process_similarity_generator(doc_db, k_val, threshold):
    """Generator to yield results one by one to save RAM."""
    names = list(doc_db.keys())
    for da, db in combinations(names, 2):
        w_a, w_b = doc_db[da], doc_db[db]
        s_a, s_b, passages = calculate_turnitin_similarity(w_a, w_b, k=k_val)
        max_s = max(s_a, s_b)
        yield {
            "Dokumen 1": da, "Dokumen 2": db, "Max Score": round(max_s, 2),
            "Status": "PASS" if max_s <= threshold else "FAIL",
            "Badge": turnitin_badge(max_s), "Score A": round(s_a, 2),
            "Score B": round(s_b, 2), "Words A": len(w_a), "Words B": len(w_b),
            "Matches": len(passages), "Passages": passages[:max_preview_passages]
        }

if 'doc_database' not in globals() or len(doc_database) < 2:
    print("❌ Minimal 2 dokumen untuk perbandingan.")
else:
    print(f"🚀 Memulai pemrosesan {math.comb(len(doc_database), 2):,} pasangan...")
    excel_file = "Turnitin_Similarity_Report_P3MD.xlsx"
    wb = openpyxl.Workbook()

    # 1. Summary Sheet
    ws1 = wb.active
    ws1.title = "Similarity Summary"
    headers = ["Dokumen 1", "Dokumen 2", "Turnitin Max Score (%)", "Status Kelulusan", "Kategori Turnitin", "Matches"]
    ws1.append(headers)

    # 2. Detail Sheet
    ws2 = wb.create_sheet("Matched Passages Detail")
    ws2.append(["Pasangan Dokumen", "Skor Max (%)", "Potongan Kalimat Identik"])

    green_fill = PatternFill(start_color="C6EFCE", end_color="C6EFCE", fill_type="solid")
    red_fill = PatternFill(start_color="FFC7CE", end_color="FFC7CE", fill_type="solid")

    # Stream processing
    count = 0
    for res in process_similarity_generator(doc_database, min_consecutive_words, PASS_THRESHOLD):
        # Add to Summary
        row_idx = ws1.max_row + 1
        ws1.append([res["Dokumen 1"], res["Dokumen 2"], res["Max Score"], res["Status"], res["Badge"], res["Matches"]])

        # Style status cell
        status_cell = ws1.cell(row=row_idx, column=4)
        status_cell.fill = green_fill if res["Status"] == "PASS" else red_fill

        # Add to Passages
        for p in res["Passages"]:
            ws2.append([f"{res['Dokumen 1']} vs {res['Dokumen 2']}", f"{res['Max Score']}%", p])

        count += 1
        if count % 1000 == 0: print(f"  > Diproses: {count} pasangan...")

    # 3. Metadata Sheet
    ws3 = wb.create_sheet("Analysis Metadata")
    ws3.append(["Parameter Name", "Value"])
    ws3.append(["Official Commander Threshold", f"{COMMANDER_THRESHOLD}%"])
    ws3.append(["Safety Cushion Threshold", f"{PASS_THRESHOLD}%"])
    ws3.append(["Min Consecutive Words", min_consecutive_words])
    ws3.append(["Total Documents", len(doc_database)])
    ws3.append(["Total Comparisons", count])

    wb.save(excel_file)
    print(f"✅ Selesai! {count} pasangan dianalisis.")
    print(f"💾 Laporan tersimpan di '{excel_file}'.")
    try:
        files.download(excel_file)
    except: pass